# Trening EmotionCNN na FER2013 (Google Colab)

Notebook trenuje model rozpoznawania emocji z wykorzystaniem darmowego GPU T4 w Colab.

**Czas treningu:** ~10-15 min na 30 epok (T4 GPU).

## Kolejnosc krokow
1. Wlacz GPU: `Runtime` -> `Change runtime type` -> `T4 GPU`
2. Uruchom komorki po kolei
3. Pobierz `emotion_model_best.pth` + wykresy na koniec
4. Skopiuj `.pth` lokalnie do `emotions-project/models/emotion_model.pth`

## 1. Sprawdzenie GPU

In [ ]:
import torch
print('CUDA dostepne:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'brak')
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Dataset FER2013

**Opcja A** - mount Google Drive (jezeli wgrales FER2013.zip do swojego Drive):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ZMIEN sciezke na swoja lokalizacje archiwum FER2013.zip
!cp '/content/drive/MyDrive/FER2013.zip' /content/FER2013.zip
!unzip -q /content/FER2013.zip -d /content/
!ls /content/FER2013

**Opcja B** - pobranie z Kaggle (wymaga `kaggle.json` z Twojego konta Kaggle):

In [ ]:
# Wgraj kaggle.json przez panel po lewej (Files), potem odkomentuj:
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d msambare/fer2013 -p /content/
# !unzip -q /content/fer2013.zip -d /content/FER2013
# !ls /content/FER2013

## 3. Kod modelu i treningu

Wklejamy `model.py` i `train.py` bezposrednio (zamiast klonowac repo).

In [ ]:
%%writefile model.py
import torch.nn as nn
import torch.nn.functional as F

class EmotionCNN(nn.Module):
    def __init__(self, num_classes):
        super(EmotionCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(128 * 6 * 6, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))
        x = x.view(-1, 128 * 6 * 6)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

Wgraj `train.py` recznie przez panel Files (ikona folderu po lewej) ALBO wklej zawartosc do komorki ponizej (`%%writefile train.py`).

Po wgraniu odpal:

In [ ]:
!ls
# Powinno byc widac: model.py, train.py, FER2013/ (lub /content/FER2013/)

## 4. Trening

Domyslnie: 30 epok, batch 64, augmentacja wlaczona, class weights, early stopping (patience=5).

In [ ]:
!python train.py \
    --data_dir /content/FER2013 \
    --output_dir /content/out \
    --epochs 30 \
    --batch_size 64 \
    --num_workers 2

## 5. Podglad wynikow

In [ ]:
from IPython.display import Image, display
display(Image('/content/out/learning_curves.png'))
display(Image('/content/out/confusion_matrix.png'))

with open('/content/out/classification_report.txt') as f:
    print(f.read())

## 6. Pobranie wynikow

In [ ]:
# Spakuj wszystko i sciagnij jeden ZIP
!cd /content/out && zip -r /content/results.zip . && ls -la /content/results.zip
from google.colab import files
files.download('/content/results.zip')

**Po pobraniu rozpakuj** i skopiuj `emotion_model_best.pth` lokalnie do `emotions-project/models/emotion_model.pth`. `detect.py` i `webapp/` zaczna od razu uzywac nowego modelu.

## 7. Eksperyment porownawczy - bez augmentacji (opcjonalnie)

Do sprawozdania warto pokazac wplyw augmentacji. Drugi run bez augmentacji:

In [ ]:
!python train.py \
    --data_dir /content/FER2013 \
    --output_dir /content/out_no_aug \
    --epochs 30 \
    --batch_size 64 \
    --num_workers 2 \
    --no_augment